# 03b — Churn Label Audit
SpiriCom · Huawei Technologies Tunisia · PFE 2026
Objective:
Detect hidden data imputation in KPI signals that directly affect
churn labeling (v5 rule-based system).

Risk:
 If churn-defining variables are artificially imputed,
 the churn label becomes statistically biased and non-causal.

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from datetime import datetime

pd.set_option('display.float_format', '{:.4f}'.format)
PROC_DIR  = Path('data/processed')
OUT_DIR   = Path('data/outputs')
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

df_kpi  = pd.read_parquet(PROC_DIR / 'kpi_clean.parquet')
df_kpi.columns = df_kpi.columns.str.lower().str.replace(' ', '_')
df_comp = pd.read_parquet(PROC_DIR / 'complaints_clean.parquet')
df_comp.columns = df_comp.columns.str.lower().str.replace(' ', '_')
print(f'KPI        : {df_kpi.shape[0]:,} rows x {df_kpi.shape[1]} cols')
print(f'Complaints : {df_comp.shape[0]:,} rows x {df_comp.shape[1]} cols')


KPI        : 4,898 rows x 122 cols
Complaints : 25,727 rows x 23 cols


## 1 — Imputation forensics (NB03-2)


In [2]:


# ────────────────────────────────────────────────────────────────
# STEP 1 — Define critical KPI variables for churn logic
# ────────────────────────────────────────────────────────────────
# These variables directly influence churn definition:
#   - dou_total     → usage intensity
#   - duration      → session engagement
#   - traffic_*     → network behavior
#   - latency KPIs  → QoE degradation indicators

IMPUTE_CHECK_COLS = [
    'dou_total', 'duration', 'traffic_5g',
    'traffic_4g', 'traffic_3g', 'traffic_2g',
    'e2e_delay_ms', 'client_rtt_ms'
]

IMPUTE_CHECK_COLS = [c for c in IMPUTE_CHECK_COLS if c in df_kpi.columns]


# ────────────────────────────────────────────────────────────────
# STEP 2 — Spike-based imputation detection
# ────────────────────────────────────────────────────────────────
# Method:
# If a single value appears in >5% of records,
# it is considered a "synthetic imputation artifact"
#
# Rationale:
# Natural KPI distributions are continuous and heterogeneous.
# A dominant repeated value usually indicates:
#   → median imputation
#   → zero-filling
#   → system fallback values

SPIKE_FRAC = 0.05   # 5% threshold for anomaly detection

imputed_flags = {}

print(f'{"column":<22s} {"spike value":>20s} {"spike rows":>11s} {"share":>8s}')

for c in IMPUTE_CHECK_COLS:

    vc = df_kpi[c].value_counts()

    if len(vc) == 0:
        continue

    spike_val = vc.index[0]
    spike_n    = int(vc.iloc[0])
    share      = spike_n / len(df_kpi)

    # ── Imputation decision rule ───────────────────────────────
    flagged = share > SPIKE_FRAC

    if flagged:
        imputed_flags[c] = spike_val

        # Create binary flag for downstream filtering / modeling
        df_kpi[f'{c}_imputed'] = (df_kpi[c] == spike_val).astype(int)

    print(
        f'{c:<22s} {spike_val:>20,.0f} {spike_n:>11,} {share*100:>7.1f}%'
        + ('  <- IMPUTED (flagged)' if flagged else '')
    )


# ────────────────────────────────────────────────────────────────
# STEP 3 — Summary of detected imputation bias
# ────────────────────────────────────────────────────────────────

print(f'\nDetected imputed columns: {list(imputed_flags)}')


# ────────────────────────────────────────────────────────────────
# STEP 4 — Impact on churn labeling integrity
# ────────────────────────────────────────────────────────────────
# If churn-defining KPIs are imputed:
#   → churn boundary becomes artificial
#   → Q20 thresholds lose statistical meaning
#   → model learns data artifacts instead of behavior

if 'dou_total' in imputed_flags or 'duration' in imputed_flags:

    print('\n  WARNING — Label integrity risk detected')
    print('CONFIRMED NB03-2: churn-defining columns contain imputed values')
    print('These records introduce bias into churn labeling logic')

    print('\n→ Recommendation:')
    print('   - treat affected rows as LOW CONFIDENCE')
    print('   - optionally exclude from supervised learning')
    print('   - or assign churn = NaN for audit-safe modeling')

column                          spike value  spike rows    share
dou_total                       231,108,281       1,358    27.7%  <- IMPUTED (flagged)
duration                                353       1,509    30.8%  <- IMPUTED (flagged)
traffic_5g                          193,498       4,498    91.8%  <- IMPUTED (flagged)
traffic_4g                      254,279,290       1,734    35.4%  <- IMPUTED (flagged)
traffic_3g                        3,746,973       2,028    41.4%  <- IMPUTED (flagged)
traffic_2g                           65,048       3,940    80.4%  <- IMPUTED (flagged)
e2e_delay_ms                            164       2,072    42.3%  <- IMPUTED (flagged)
client_rtt_ms                           118       2,092    42.7%  <- IMPUTED (flagged)

Detected imputed columns: ['dou_total', 'duration', 'traffic_5g', 'traffic_4g', 'traffic_3g', 'traffic_2g', 'e2e_delay_ms', 'client_rtt_ms']

  WARNING — Label integrity risk detected
CONFIRMED NB03-2: churn-defining columns contain imput

## 2 — MSISDN linkage rescue attempt (NB03-3)

In [3]:
# Objective:
# Evaluate whether complaint dataset (df_comp) and KPI dataset (df_kpi)
# can be linked at customer level despite anonymization.
#
# Risk:
# Telecom identifiers (MSISDN) are often:
#   → salted (hashing)
#   → partially masked
#   → format-normalized differently per system
#
# Therefore, exact joins are not reliable → we test fuzzy matching.
# ══════════════════════════════════════════════════════════════════════


# ────────────────────────────────────────────────────────────────
# STEP 1 — Normalize MSISDN representations
# ────────────────────────────────────────────────────────────────
# Goal:
# Generate multiple transformation hypotheses to recover alignment:
#
#   1. raw         → original identifier
#   2. lstrip_zero → removes leading zeros
#   3. last8       → last 8 digits (common telecom fallback key)
#   4. digits_only  → remove formatting noise (+, spaces, etc.)

def norm_variants(s):
    s = s.dropna().astype(str).str.strip()

    # remove invalid identifiers (noise, placeholders)
    s = s[s.str.len() >= 4]

    # extract only numeric content
    digits = s.str.replace(r'\D', '', regex=True)

    return {
        'raw': set(s),
        'lstrip_zero': set(s.str.lstrip('0')),
        'last8': set(digits.str[-8:]),
        'digits_only': set(digits),
    }


# Apply normalization strategies
v_comp = norm_variants(df_comp['msisdn'])
v_kpi  = norm_variants(df_kpi['msisdn'])


# ────────────────────────────────────────────────────────────────
# STEP 2 — Matching feasibility analysis
# ────────────────────────────────────────────────────────────────
# We evaluate overlap between both datasets under each strategy

print(f'{"strategy":<14s} {"comp":>10s} {"kpi":>10s} {"overlap":>10s} {"rate":>8s}')

best_strategy, best_overlap = None, -1

for k in v_comp:

    # intersection between transformed identifier spaces
    ov = len(v_comp[k] & v_kpi[k])

    # match rate relative to KPI population
    rate = ov / max(len(v_kpi[k]), 1)

    print(f'{k:<14s} {len(v_comp[k]):>10,} {len(v_kpi[k]):>10,} '
          f'{ov:>10,} {rate*100:>7.1f}%')

    # track best possible linkage strategy
    if ov > best_overlap:
        best_strategy, best_overlap = k, ov


# ────────────────────────────────────────────────────────────────
# STEP 3 — Decision rule: linkability assessment
# ────────────────────────────────────────────────────────────────
# A dataset is considered linkable if:
#   overlap ≥ 5% of unique KPI customers
#
# This is a conservative threshold used in telecom identity recovery.

LINKABLE = best_overlap >= 0.05 * df_kpi['msisdn'].nunique()


print(f'\nBest strategy: {best_strategy} ({best_overlap:,} matches)')


# ────────────────────────────────────────────────────────────────
# STEP 4 — Interpretation of results
# ────────────────────────────────────────────────────────────────

if LINKABLE:

    print('\n✅ Linkage POSSIBLE after normalization')
    print('Complaint features can be integrated into NB04 feature engineering')
    print('using the identified transformation strategy.')

else:

    print('\n❌ Linkage NOT POSSIBLE under all tested transformations')

    print('\nCONFIRMED FINDING:')
    print('- MSISDN identifiers are inconsistently anonymized')
    print('- Likely different hashing salts or encryption pipelines')
    print('- Cross-dataset customer-level join is not statistically valid')

    print('\nMethodological consequence:')
    print('→ Complaint dataset must remain independent (NB01)')
    print('→ NB04/NB05 modeling will be KPI-only (no leakage risk)')

strategy             comp        kpi    overlap     rate
raw                22,125      4,890          1     0.0%
lstrip_zero        22,085      4,890          1     0.0%
last8              22,123      4,890          1     0.0%
digits_only        22,125      4,890          1     0.0%

Best strategy: raw (1 matches)

❌ Linkage NOT POSSIBLE under all tested transformations

CONFIRMED FINDING:
- MSISDN identifiers are inconsistently anonymized
- Likely different hashing salts or encryption pipelines
- Cross-dataset customer-level join is not statistically valid

Methodological consequence:
→ Complaint dataset must remain independent (NB01)
→ NB04/NB05 modeling will be KPI-only (no leakage risk)


## 3 — Label : thresholds on observed data, imputed rows unlabelled

In [4]:
# Objective:
# Construct a statistically valid churn label by:
#   1. excluding imputed KPI records from threshold estimation
#   2. separating label-observable vs unobservable customers
#   3. preventing artificial bias introduced by data imputation
#
# Key idea:
# → churn definition is only reliable on "observed data region"
# → imputed rows are treated as unlabeled (semi-supervised setting)
# ══════════════════════════════════════════════════════════════════════


# ────────────────────────────────────────────────────────────────
# STEP 1 — Build aggregation at MSISDN level
# ────────────────────────────────────────────────────────────────
# We reconstruct customer-level behavior from raw KPI logs

agg = dict(
    session_flag=('session_flag', 'max'),
    traffic_5g=('traffic_5g', 'sum'),
    dou_total=('dou_total', 'sum'),
    duration=('duration', 'sum'),
    brand=('brand', 'first'),
)

# Preserve imputation indicators (if available)
for c in imputed_flags:
    if f'{c}_imputed' in df_kpi.columns:
        agg[f'{c}_imputed'] = (f'{c}_imputed', 'max')

churn = df_kpi.groupby('msisdn').agg(**agg).reset_index()


# ────────────────────────────────────────────────────────────────
# STEP 2 — Identify label-observable population
# ────────────────────────────────────────────────────────────────
# A customer is label-valid if:
#   → no imputation on churn-defining KPIs (DOU, duration)

dou_imp = churn.get('dou_total_imputed', pd.Series(0, index=churn.index))
dur_imp = churn.get('duration_imputed',  pd.Series(0, index=churn.index))

label_observed = (dou_imp == 0) & (dur_imp == 0)

n_obs = int(label_observed.sum())

print(f'Customers total                  : {len(churn):,}')
print(f'Label-observable (clean KPIs)    : {n_obs:,} '
      f'({n_obs/len(churn)*100:.1f}%)')


# ────────────────────────────────────────────────────────────────
# STEP 3 — Statistical threshold estimation (clean subset only)
# ────────────────────────────────────────────────────────────────
# Important methodological constraint:
# → Q20 thresholds are computed ONLY on observed data
# → avoids bias from imputed distributions

obs = churn[label_observed]

thr_dou = obs['dou_total'].quantile(0.20)
thr_dur = obs['duration'].quantile(0.20)

print(f'\nDOU Q20 (observed only) : {thr_dou:>15,.0f} bytes ({thr_dou/1e6:.2f} MB)')
print(f'Dur Q20 (observed only) : {thr_dur:>15,.0f} seconds')


# ────────────────────────────────────────────────────────────────
# STEP 4 — Rule-based label construction (v6)
# ────────────────────────────────────────────────────────────────
# Churn is defined as behavioral disengagement:
#   C1: low data usage (≤ Q20)
#   C2: low session duration (≤ Q20)
#
# Final rule:
#   churn = C1 OR C2 (only if observed)

churn['c1_low_usage'] = (churn['dou_total'] <= thr_dou).astype(int)
churn['c2_low_dur']   = (churn['duration']  <= thr_dur).astype(int)

churn['churn'] = np.where(
    label_observed,
    (
        (churn['c1_low_usage'] == 1) |
        (churn['c2_low_dur'] == 1)
    ).astype(float),
    np.nan  # imputed customers → unlabeled (no ground truth)
)


# ────────────────────────────────────────────────────────────────
# STEP 5 — Final label distribution (observed population only)
# ────────────────────────────────────────────────────────────────

lab = churn['churn'].dropna()
rate = lab.mean()

print(f'\n=== DISENGAGEMENT LABEL v6 (OBSERVED-ONLY) ===')
print(f'Labelled population : {len(lab):,}')
print(f'Disengaged          : {int(lab.sum()):,}')
print(f'Engaged             : {int((lab == 0).sum()):,}')
print(
    f'Disengaged share    : {rate*100:.1f}% '
    '(rule-based definition, NOT empirical churn rate)'
)
print(f'Unlabelled (imputed): {int(churn["churn"].isna().sum()):,}')


# ────────────────────────────────────────────────────────────────
# STEP 6 — Structural validation checks
# ────────────────────────────────────────────────────────────────
# Ensures mathematical consistency of label construction

assert len(churn) > 0 and len(lab) > 0
assert thr_dou > 0 and thr_dur > 0
assert 0 < rate < 1

print('\n Structural assertions passed')

Customers total                  : 4,896
Label-observable (clean KPIs)    : 2,566 (52.4%)

DOU Q20 (observed only) :       1,856,088 bytes (1.86 MB)
Dur Q20 (observed only) :              82 seconds

=== DISENGAGEMENT LABEL v6 (OBSERVED-ONLY) ===
Labelled population : 2,566
Disengaged          : 868
Engaged             : 1,698
Disengaged share    : 33.8% (rule-based definition, NOT empirical churn rate)
Unlabelled (imputed): 2,330

 Structural assertions passed


## 4 — Leakage blocklist (NB03-1)

In [5]:
# Objective:
# Prevent data leakage in machine learning models by explicitly
# removing features that are mathematically or structurally derived
# from the churn label definition.
#
# Key principle:
# A feature is considered "leaky" if it directly or indirectly
# encodes the churn decision rule:
#
#   churn = f(dou_total, duration)
#
# Therefore:
#   → all volume / usage / duration KPIs must be excluded
# ══════════════════════════════════════════════════════════════════════


# ────────────────────────────────────────────────────────────────
# STEP 1 — Identify volume-based feature family
# ────────────────────────────────────────────────────────────────
# These features represent raw usage intensity signals

VOLUME_FAMILY = [
    c for c in df_kpi.columns
    if c.endswith('_traffic')
]


# ────────────────────────────────────────────────────────────────
# STEP 2 — Construct leakage blocklist
# ────────────────────────────────────────────────────────────────
# These features are excluded because they are:
#   (1) directly used in churn definition
#   (2) deterministic transformations of churn rule variables
#   (3) proxies for behavioral intensity (same information as label)

LEAKAGE_BLOCKLIST = sorted(
    set(
        [
            'dou_total',
            'duration',
            'session_flag',

            # multi-generational traffic KPIs
            'traffic_2g', 'traffic_3g', 'traffic_4g', 'traffic_5g',

            # voice usage KPIs
            'voice_onlinetime_2g', 'voice_onlinetime_3g',

            # temporal usage distribution
            'night_traffic', 'day_traffic', 'late_night_traffic',

            # label construction intermediates
            'c1_low_usage',
            'c2_low_dur',
        ]
        + VOLUME_FAMILY
    )
    & set(list(df_kpi.columns) + ['c1_low_usage', 'c2_low_dur'])
)


# ────────────────────────────────────────────────────────────────
# STEP 3 — Define safe (non-leaky) feature candidates
# ────────────────────────────────────────────────────────────────
# These represent network quality, device capability, and context,
# which are NOT used in churn definition

ALLOWED_EXAMPLES = [
    c for c in [
        'e2e_delay_ms',
        'client_rtt_ms',
        'server_rtt_ms',

        'client_packet_loss_rate',
        'server_packet_loss_rate',

        'page_response_delay',
        'page_download_throughput',

        'dns_delay',
        'dns_sr',
        'tcp_connection_sr',

        # categorical / contextual features
        'brand',
        'generation',
        'tertype',
        'sim_capability',
        'usim_flag',
        'usertype',
        'user_class',
        'mobility_class',
        'number_of_regions',

        # geographic segmentation
        'layer2name',
        'layer3name',
    ]
    if c in df_kpi.columns
]


# ────────────────────────────────────────────────────────────────
# STEP 4 — Export leakage control manifest
# ────────────────────────────────────────────────────────────────
# This file ensures reproducibility and auditability of ML pipeline

blocklist_payload = {
    'generated_at': datetime.now().isoformat(),

    # methodological justification
    'reason': (
        'churn v6 label is a deterministic function of dou_total/duration; '
        'therefore all volume and duration-based KPIs are considered '
        'data leakage and must be excluded from predictive modeling'
    ),

    # hard exclusion list for ML pipeline
    'blocked_features': LEAKAGE_BLOCKLIST,

    # safe feature space examples
    'allowed_examples': ALLOWED_EXAMPLES,

    # reference thresholds (for interpretability only)
    'thresholds': {
        'dou_q20_bytes': float(thr_dou),
        'dur_q20_seconds': float(thr_dur),
    },
}


# ────────────────────────────────────────────────────────────────
# STEP 5 — Save artifact for modeling pipeline
# ────────────────────────────────────────────────────────────────

bl_path = MODEL_DIR / 'leakage_blocklist.json'

with open(bl_path, 'w') as f:
    json.dump(blocklist_payload, f, indent=2)


# ────────────────────────────────────────────────────────────────
# STEP 6 — Audit output
# ────────────────────────────────────────────────────────────────

print(f'Saved: {bl_path}')

print(f'Blocked features ({len(LEAKAGE_BLOCKLIST)}):')
print(LEAKAGE_BLOCKLIST)

print(f'\nAllowed feature examples ({len(ALLOWED_EXAMPLES)}):')
print(ALLOWED_EXAMPLES)

Saved: models\leakage_blocklist.json
Blocked features (31):
['c1_low_usage', 'c2_low_dur', 'day_traffic', 'dou_total', 'duration', 'facebook_messenger_traffic', 'facebook_traffic', 'game_traffic', 'google_common_traffic', 'googlesearch_traffic', 'https_traffic', 'im_traffic', 'instagram_traffic', 'late_night_traffic', 'night_traffic', 'other_traffic', 'quic_ietf_traffic', 'session_flag', 'sms_traffic', 'streaming_traffic', 'tiktok_traffic', 'traffic_2g', 'traffic_3g', 'traffic_4g', 'traffic_5g', 'voice_onlinetime_2g', 'voice_onlinetime_3g', 'voip_traffic', 'web_browsing_traffic', 'whatsapp_traffic', 'youtube_traffic']

Allowed feature examples (20):
['e2e_delay_ms', 'client_rtt_ms', 'server_rtt_ms', 'client_packet_loss_rate', 'server_packet_loss_rate', 'page_response_delay', 'page_download_throughput', 'dns_delay', 'dns_sr', 'tcp_connection_sr', 'brand', 'generation', 'tertype', 'usim_flag', 'usertype', 'user_class', 'mobility_class', 'number_of_regions', 'layer2name', 'layer3name']


## 5 — Export `churn_labelled_v6.parquet` + corrected JSON

In [6]:
# Attach modelling-safe context columns
for col in ['generation', 'layer2name', 'mobility_class', 'usertype',
            'user_class', 'tertype']:
    if col in df_kpi.columns and col not in churn.columns:
        m = df_kpi.drop_duplicates('msisdn')[['msisdn', col]].copy()
        m[col] = (m[col].astype(str).str.strip().str.upper()
                  .replace({'NAN': 'UNKNOWN', '': 'UNKNOWN'}))
        churn = churn.merge(m, on='msisdn', how='left')

QOS_COLS = [c for c in [
    'e2e_delay_ms', 'client_rtt_ms', 'server_rtt_ms',
    'client_packet_loss_rate', 'server_packet_loss_rate',
    'page_response_delay', 'page_download_throughput',
    'dns_delay', 'dns_sr', 'tcp_connection_sr', 'number_of_regions',
] if c in df_kpi.columns]
if QOS_COLS:
    qos = df_kpi.groupby('msisdn')[QOS_COLS].mean().reset_index()
    churn = churn.merge(qos, on='msisdn', how='left')

out_path = PROC_DIR / 'churn_labelled_v6.parquet'
churn.to_parquet(out_path, index=False)
print(f'Saved: {out_path}  ({churn.shape[0]:,} rows x {churn.shape[1]} cols)')
print('churn column: 1.0 disengaged / 0.0 engaged / NaN unlabelled (imputed)')

lab = churn['churn'].dropna()
eda_json = {
    'generated_at'        : datetime.now().isoformat(),
    'label_version'       : 'v6 - observed-only thresholds, imputed rows unlabelled',
    'definition_note'     : 'disengagement segment (dou<=Q20 OR duration<=Q20), '
                            'a design parameter - NOT a measured churn rate',
    'total_customers'     : int(len(churn)),
    'labelled_customers'  : int(len(lab)),
    'unlabelled_imputed'  : int(churn['churn'].isna().sum()),
    'disengaged'          : int(lab.sum()),
    'engaged'             : int((lab == 0).sum()),
    'disengaged_share_pct': round(float(lab.mean()) * 100, 2),
    'thresholds'          : {'dou_q20_bytes': int(thr_dou),
                             'dur_q20_seconds': int(thr_dur)},
    'msisdn_linkage'      : {'best_strategy': best_strategy,
                             'best_overlap': int(best_overlap),
                             'linkable': bool(LINKABLE)},
    'leakage_blocklist'   : str(bl_path),
    'saved_file'          : str(out_path),
}
jp = OUT_DIR / 'churn_eda_v6.json'
json.dump(eda_json, open(jp, 'w'), indent=2, default=str)
print(f'Saved: {jp}')
print(json.dumps(eda_json, indent=2, default=str))
print('\nNext -> NB04 must: load churn_labelled_v6.parquet, drop NaN labels,')
print('apply models/leakage_blocklist.json, and engineer features ONLY from')
print('the allowed families (QoS / device / geography / mobility).')


Saved: data\processed\churn_labelled_v6.parquet  (4,896 rows x 34 cols)
churn column: 1.0 disengaged / 0.0 engaged / NaN unlabelled (imputed)
Saved: data\outputs\churn_eda_v6.json
{
  "generated_at": "2026-06-30T09:24:15.447987",
  "label_version": "v6 - observed-only thresholds, imputed rows unlabelled",
  "definition_note": "disengagement segment (dou<=Q20 OR duration<=Q20), a design parameter - NOT a measured churn rate",
  "total_customers": 4896,
  "labelled_customers": 2566,
  "unlabelled_imputed": 2330,
  "disengaged": 868,
  "engaged": 1698,
  "disengaged_share_pct": 33.83,
  "thresholds": {
    "dou_q20_bytes": 1856088,
    "dur_q20_seconds": 82
  },
  "msisdn_linkage": {
    "best_strategy": "raw",
    "best_overlap": 1,
    "linkable": false
  },
  "leakage_blocklist": "models\\leakage_blocklist.json",
  "saved_file": "data\\processed\\churn_labelled_v6.parquet"
}

Next -> NB04 must: load churn_labelled_v6.parquet, drop NaN labels,
apply models/leakage_blocklist.json, and en